# 06 — Feature Engineering

## Objectif de l'étude

L'objectif de ce notebook est d'évaluer si une meilleure représentation des rendements passés permet d'obtenir un signal prédictif plus informatif que les rendements bruts utilisés jusqu'à présent.

Dans le notebook précédent, la régression logistique entraînée directement sur `RET_1` à `RET_20` a obtenu des performances très proches des baselines déterministes. Ce résultat suggère que la limite provient peut-être moins du modèle que de la représentation des données.

Afin d'isoler uniquement l'effet du feature engineering, le modèle de régression logistique, le protocole de validation temporelle et les métriques d'évaluation restent strictement identiques aux étapes précédentes.

## Protocole expérimental

Toutes les expériences réalisées dans ce notebook suivent exactement le même protocole que celui défini précédemment.

Les éléments suivants restent inchangés :

- la stratégie de validation temporelle par fenêtres expansives ;
- le pipeline de régression logistique ;
- les fonctions d'évaluation ;
- les métriques utilisées : Accuracy, ROC-AUC et Log-Loss.

Ainsi, toute différence de performance observée pourra être attribuée au feature engineering et non à une modification du modèle ou du protocole expérimental.

## Hypothèse 1 — Momentum multi-horizon

### Hypothèse financière

Le comportement d'une allocation dépend potentiellement de l'horizon temporel sur lequel on observe ses rendements passés.

Un momentum calculé sur quelques jours résume la dynamique récente du marché, tandis qu'un momentum calculé sur une période plus longue décrit une tendance plus globale.

Notre hypothèse est qu'une représentation agrégée des rendements sur plusieurs horizons temporels pourrait capturer des informations que les rendements bruts pris individuellement ne mettent pas suffisamment en évidence.

Cette famille de features permettra également d'étudier si le comportement récent confirme, ralentit ou diverge par rapport à la tendance observée sur un horizon plus long.

## Analyse du risque de fuite d'information (Leakage)

Les variables créées dans cette première famille de features sont calculées uniquement à partir des colonnes `RET_1` à `RET_20`.

Ces rendements correspondent exclusivement à des observations historiques disponibles au moment de la prédiction.

Aucune information future ni aucune variable cible (`LABEL`) n'intervient dans leur calcul.

Par conséquent, cette famille de features respecte entièrement la contrainte de temporalité et n'introduit aucun risque de fuite d'information (target leakage).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import load_X_train, load_y_train
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.modeling import build_logistic_regression_pipeline
from src.evaluation import evaluate_model_on_folds, compare_model_results
from src.features import (create_momentum_features, 
                          create_volatility_features, 
                          create_momentum_volatility_ratio_features,
                          create_volume_features)

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train = X_train.merge(y_train,left_on="ROW_ID",right_on="ROW_ID")
df_train["class"] = np.where(df_train["target"] > 0,1,0)

In [5]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

## Création des features de momentum

La première famille de features consiste à résumer les rendements passés sur plusieurs horizons temporels.

Les variables créées sont :

- `momentum_3` : moyenne des trois derniers rendements ;
- `momentum_5` : moyenne des cinq derniers rendements ;
- `momentum_10` : moyenne des dix derniers rendements ;
- `momentum_20` : moyenne des vingt derniers rendements.

Une variable supplémentaire est également créée :

- `delta_momentum_3_20`, qui mesure la différence entre le momentum de très court terme et le momentum de plus long terme.

Cette dernière feature vise à quantifier la divergence entre la dynamique récente et la tendance observée sur un horizon plus long.

In [6]:
df_train_momentum = create_momentum_features(df_train)

In [7]:
df_train_momentum.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'momentum_3', 'momentum_5', 'momentum_10',
       'momentum_20', 'delta_momentum_3_20'],
      dtype='str')

In [8]:
momentum_features = ['momentum_3', 'momentum_5', 'momentum_10', 'momentum_20', 'delta_momentum_3_20']

raw_return_features  = ['RET_20', 'RET_19', 'RET_18', 'RET_17','RET_16', 
                 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 
                 'RET_10','RET_9', 'RET_8', 'RET_7', 'RET_6', 
                 'RET_5', 'RET_4', 'RET_3', 'RET_2','RET_1'
                ]

In [9]:
logistic_regression_pipeline = build_logistic_regression_pipeline()

## Évaluation de la première hypothèse

Nous évaluons maintenant la régression logistique en utilisant uniquement les features de momentum créées précédemment.

L'objectif n'est pas encore d'obtenir les meilleures performances possibles, mais de déterminer si cette nouvelle représentation des rendements historiques apporte un signal plus informatif que les rendements bruts.

Les résultats seront ensuite comparés :

- aux baselines déterministes ;
- à la régression logistique entraînée sur `RET_1` à `RET_20`.

In [10]:
logistic_regression_momentum_results = evaluate_model_on_folds(df_train_momentum,folds,logistic_regression_pipeline,momentum_features)
logistic_regression_raw_results = evaluate_model_on_folds(df_train,folds,logistic_regression_pipeline,raw_return_features )

## Comparaison avec la régression logistique brute

Nous comparons maintenant la régression logistique entraînée sur les features de momentum avec la régression logistique de référence entraînée sur les rendements bruts `RET_1` à `RET_20`.

Le modèle, les folds et les métriques restent identiques. La seule différence est la représentation des variables explicatives.

In [11]:
comparison_momentum_vs_raw = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_momentum_results,
    model_a_name="raw",
    model_b_name="momentum",
)

comparison_momentum_vs_raw

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_momentum,log_loss_momentum,roc_auc_momentum,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.526084,0.691592,0.533467,0.009497,-0.000575,0.012532
1,2,0.529923,0.690788,0.540789,0.514306,0.692544,0.518737,-0.015617,0.001756,-0.022052
2,3,0.515708,0.692314,0.517971,0.517041,0.692440,0.515119,0.001333,0.000126,-0.002852
3,4,0.519447,0.691120,0.528017,0.506157,0.693040,0.501265,-0.013289,0.001920,-0.026751


In [12]:
comparison_summary = comparison_momentum_vs_raw.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary

accuracy_raw         0.520416
log_loss_raw         0.691597
roc_auc_raw          0.526928
accuracy_momentum    0.515897
log_loss_momentum    0.692404
roc_auc_momentum     0.517147
delta_accuracy      -0.004519
delta_log_loss       0.000807
delta_roc_auc       -0.009781
dtype: float64

## Interprétation des résultats

### Famille de features

Momentum multi-horizon (`momentum_3`, `momentum_5`, `momentum_10`, `momentum_20` et `delta_momentum_3_20`).

### Hypothèse

L'hypothèse testée est qu'une représentation agrégée des rendements passés sur plusieurs horizons temporels capture mieux la dynamique d'une allocation que les rendements bruts pris individuellement.

En particulier, nous supposons que des horizons différents permettent de résumer des comportements de marché distincts (court terme, moyen terme et divergence entre les deux) et qu'ils peuvent fournir un signal plus stable à une régression logistique.

### Résultats observés

La comparaison avec la régression logistique entraînée sur les rendements bruts montre des écarts relativement faibles.

Les features de momentum améliorent légèrement certains folds mais dégradent également les performances sur d'autres périodes. Les gains observés en Accuracy, ROC-AUC et Log-Loss restent limités et ne sont pas homogènes sur l'ensemble de la validation temporelle.

Ces résultats suggèrent que la représentation par momentum ne permet pas, à elle seule, d'extraire un signal prédictif nettement supérieur aux rendements historiques bruts.

### Décision

**Décision : Reformuler.**

La famille de features n'est pas rejetée car elle repose sur une hypothèse financière cohérente et produit des performances proches du modèle de référence.

En revanche, elle n'apporte pas une amélioration suffisamment stable pour être conservée dans sa forme actuelle.

### Justification

Plusieurs momentum calculés sur des horizons proches peuvent contenir une information fortement redondante.

La régression logistique reçoit donc plusieurs représentations très similaires de la même dynamique de marché, ce qui limite probablement l'apport de cette première famille de features.

Une approche plus pertinente pourrait consister à sélectionner uniquement les horizons les plus informatifs ou à construire des variables représentant explicitement les différences entre horizons temporels.

### Risques restants

À ce stade, il n'est pas possible de conclure que l'hypothèse financière est fausse.

Les performances limitées peuvent provenir :

- d'une forte corrélation entre les features de momentum ;
- d'une représentation encore insuffisamment discriminante ;
- de la nature linéaire de la régression logistique.

Des expériences complémentaires seront nécessaires afin de tester des représentations plus informatives de cette même hypothèse, tout en conservant le même protocole de validation temporelle.

In [13]:
momentum_features_2 = ['momentum_3', 'momentum_20', 'delta_momentum_3_20']
logistic_regression_momentum_results_2 = evaluate_model_on_folds(df_train_momentum,folds,logistic_regression_pipeline,momentum_features_2)

comparison_momentum_vs_raw_2 = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_momentum_results_2,
    model_a_name="raw",
    model_b_name="momentum",
)

comparison_momentum_vs_raw_2

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_momentum,log_loss_momentum,roc_auc_momentum,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.521398,0.692137,0.522176,0.004810,-0.000030,0.001241
1,2,0.529923,0.690788,0.540789,0.517093,0.692517,0.525313,-0.012830,0.001729,-0.015476
2,3,0.515708,0.692314,0.517971,0.510459,0.692810,0.504495,-0.005250,0.000496,-0.013476
3,4,0.519447,0.691120,0.528017,0.506703,0.692921,0.499796,-0.012744,0.001800,-0.028221


In [14]:
comparison_summary_2 = comparison_momentum_vs_raw_2.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary_2

accuracy_raw         0.520416
log_loss_raw         0.691597
roc_auc_raw          0.526928
accuracy_momentum    0.513913
log_loss_momentum    0.692596
roc_auc_momentum     0.512945
delta_accuracy      -0.006503
delta_log_loss       0.000999
delta_roc_auc       -0.013983
dtype: float64

L’hypothèse momentum multi-horizon est cohérente économiquement,
mais elle ne produit pas un signal plus robuste que les rendements bruts
avec une régression logistique.

### Décision finale sur le momentum multi-horizon

La famille de features momentum multi-horizon est reformulée puis testée sous une version plus compacte : `momentum_3`, `momentum_20` et `delta_momentum_3_20`.

Cette version vise à réduire la redondance entre horizons proches tout en conservant l'information économique principale : court terme, long terme et divergence entre les deux.

Les résultats restent cependant faibles et instables. La version compacte ne parvient pas à améliorer de manière robuste la régression logistique sur rendements bruts.

**Décision : rejeter cette famille de features dans sa forme actuelle.**

L'hypothèse financière reste plausible, mais sa représentation actuelle ne fournit pas un signal suffisamment stable pour être conservée comme amélioration du benchmark.

## Hypothèse 2 — Volatilité réalisée multi-horizon

In [15]:
df_train_volatility = create_volatility_features(df_train)
df_train_volatility.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'volatility_3', 'volatility_20',
       'delta_volatility_3_20'],
      dtype='str')

In [16]:
volatility_features = ['volatility_3', 'volatility_20', 'delta_volatility_3_20']
logistic_regression_volatility_results = evaluate_model_on_folds(df_train_volatility,folds,logistic_regression_pipeline,volatility_features)

In [17]:
comparison_volatility_vs_raw = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_volatility_results,
    model_a_name="raw",
    model_b_name="volatility",
)

comparison_volatility_vs_raw

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_volatility,log_loss_volatility,roc_auc_volatility,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.514431,0.692844,0.498494,-0.002156,0.000677,-0.022441
1,2,0.529923,0.690788,0.540789,0.501148,0.693239,0.497264,-0.028775,0.002451,-0.043525
2,3,0.515708,0.692314,0.517971,0.519141,0.692830,0.490566,0.003432,0.000516,-0.027405
3,4,0.519447,0.691120,0.528017,0.520616,0.692654,0.501783,0.001169,0.001534,-0.026234


In [18]:
comparison_summary_volatility_raw = comparison_volatility_vs_raw.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary_volatility_raw

accuracy_raw           0.520416
log_loss_raw           0.691597
roc_auc_raw            0.526928
accuracy_volatility    0.513834
log_loss_volatility    0.692892
roc_auc_volatility     0.497026
delta_accuracy        -0.006583
delta_log_loss         0.001295
delta_roc_auc         -0.029901
dtype: float64

## Hypothèse 3 — Ratio Momentum / Volatilité

### Hypothèse financière

Le momentum mesure la direction moyenne des rendements passés, tandis que la volatilité mesure le bruit ou l'incertitude autour de cette direction.

Une même valeur de momentum peut être plus ou moins fiable selon le niveau de volatilité associé.

Nous testons donc si un ratio entre momentum et volatilité permet de capturer la qualité du signal directionnel, c'est-à-dire la quantité de direction obtenue par unité de bruit.

In [19]:
df_train_volatility["volatility_20"].describe()

count    527073.000000
mean          0.002710
std           0.001411
min           0.000377
25%           0.001782
50%           0.002383
75%           0.003271
max           0.030741
Name: volatility_20, dtype: float64

In [20]:
df_train_momentum_volatility = create_momentum_features(df_train_volatility)

In [21]:
df_train_momentum_volatility = create_momentum_volatility_ratio_features(df_train_momentum_volatility)

In [22]:
df_train_momentum_volatility.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'volatility_3', 'volatility_20',
       'delta_volatility_3_20', 'momentum_3', 'momentum_5', 'momentum_10',
       'momentum_20', 'delta_momentum_3_20', 'momentum_volatility_ratio_20'],
      dtype='str')

In [23]:
momentum_volatility_features = [
    'volatility_3', 'volatility_20', 
    'momentum_3', 'momentum_20',
    'momentum_volatility_ratio_20'
]


In [24]:
logistic_regression_momentum_volatility_results = evaluate_model_on_folds(
    df_train_momentum_volatility,
    folds,
    logistic_regression_pipeline,
    momentum_volatility_features
)

In [25]:
comparison_momentum_volatility_vs_raw = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_momentum_volatility_results,
    model_a_name="raw",
    model_b_name="momentum_volatility",
)

comparison_momentum_volatility_vs_raw

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_momentum_volatility,log_loss_momentum_volatility,roc_auc_momentum_volatility,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.517542,0.692328,0.516934,0.000954,0.000161,-0.004001
1,2,0.529923,0.690788,0.540789,0.518323,0.692191,0.525499,-0.011600,0.001403,-0.015290
2,3,0.515708,0.692314,0.517971,0.510782,0.692622,0.509161,-0.004927,0.000308,-0.008810
3,4,0.519447,0.691120,0.528017,0.501910,0.693346,0.495539,-0.017537,0.002226,-0.032478


In [26]:
comparison_summary_momentum_volatility_raw = comparison_momentum_volatility_vs_raw.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary_momentum_volatility_raw

accuracy_raw                    0.520416
log_loss_raw                    0.691597
roc_auc_raw                     0.526928
accuracy_momentum_volatility    0.512139
log_loss_momentum_volatility    0.692622
roc_auc_momentum_volatility     0.511783
delta_accuracy                 -0.008277
delta_log_loss                  0.001025
delta_roc_auc                  -0.015145
dtype: float64

testons le ratio seul

In [27]:
momentum_volatility_ratio_features = ['momentum_volatility_ratio_20']

logistic_regression_momentum_volatility_ratio_results = evaluate_model_on_folds(
    df_train_momentum_volatility,
    folds,
    logistic_regression_pipeline,
    momentum_volatility_ratio_features
)

comparison_momentum_volatility_ratio_vs_raw = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_momentum_volatility_ratio_results,
    model_a_name="raw",
    model_b_name="momentum_volatility_ratio",
)

comparison_momentum_volatility_ratio_vs_raw

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_momentum_volatility_ratio,log_loss_momentum_volatility_ratio,roc_auc_momentum_volatility_ratio,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.520237,0.692291,0.517665,0.003649,0.000124,-0.003270
1,2,0.529923,0.690788,0.540789,0.513240,0.692231,0.524741,-0.016683,0.001444,-0.016048
2,3,0.515708,0.692314,0.517971,0.514618,0.692572,0.512164,-0.001090,0.000259,-0.005807
3,4,0.519447,0.691120,0.528017,0.504560,0.693100,0.497789,-0.014887,0.001980,-0.030228


In [28]:
comparison_summary_momentum_volatility_ratio_raw = comparison_momentum_volatility_ratio_vs_raw.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary_momentum_volatility_ratio_raw

accuracy_raw                          0.520416
log_loss_raw                          0.691597
roc_auc_raw                           0.526928
accuracy_momentum_volatility_ratio    0.513164
log_loss_momentum_volatility_ratio    0.692549
roc_auc_momentum_volatility_ratio     0.513089
delta_accuracy                       -0.007253
delta_log_loss                        0.000952
delta_roc_auc                        -0.013838
dtype: float64

## Hypothèse 4 — Signed Volume multi-horizon

### Hypothèse financière

Les rendements décrivent la direction passée d'une allocation, mais ne donnent pas directement d'information sur le niveau d'activité de marché associé à ce mouvement.

Le `SIGNED_VOLUME` apporte une information complémentaire sur le contexte de marché de l'allocation. Il peut renseigner sur le régime de liquidité, l'intensité récente des flux et les changements éventuels d'activité autour de l'allocation.

Nous faisons l'hypothèse qu'une variation récente du signed volume par rapport à son régime long terme peut contenir une information exploitable par le modèle.

Cette hypothèse est testée avec trois features simples :

- `volume_3` : activité moyenne récente ;
- `volume_20` : régime moyen de volume sur un horizon plus long ;
- `delta_volume_3_20` : différence entre activité récente et régime long terme.

### Analyse du risque de fuite d'information

Les features de volume sont calculées uniquement à partir des colonnes `SIGNED_VOLUME_1` à `SIGNED_VOLUME_20`.

Ces colonnes représentent des informations historiques disponibles au moment de la prédiction.

Aucune information future, aucune donnée issue de la cible et aucune statistique calculée sur la période de validation future ne sont utilisées.

Cette famille de features respecte donc la contrainte de temporalité, sous réserve que les colonnes `SIGNED_VOLUME_i` soient bien définies comme des observations passées.

### Protocole d'évaluation

Comme pour les familles précédentes, le protocole expérimental reste inchangé :

- mêmes folds temporels expanding-window ;
- même pipeline de régression logistique ;
- mêmes métriques : Accuracy, ROC-AUC et Log-Loss ;
- comparaison contre la régression logistique sur rendements bruts.

L'objectif est d'isoler l'apport informationnel de la famille `SIGNED_VOLUME`, sans modifier le modèle ni la validation.

In [29]:
df_train_volumes = create_volume_features(df_train)
df_train_volumes.columns

Index(['ROW_ID', 'TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17',
       'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10',
       'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2',
       'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP',
       'target', 'class', 'volume_3', 'volume_20', 'delta_volume_3_20'],
      dtype='str')

In [30]:
volume_features = ['volume_3', 'volume_20', 'delta_volume_3_20']


In [31]:
logistic_regression_volume_results = evaluate_model_on_folds(
    df_train_volumes,
    folds,
    logistic_regression_pipeline,
    volume_features
)

In [32]:
comparison_volume_vs_raw = compare_model_results(
    logistic_regression_raw_results,
    logistic_regression_volume_results,
    model_a_name="raw",
    model_b_name="volume",
)

comparison_volume_vs_raw

,fold,accuracy_raw,log_loss_raw,roc_auc_raw,accuracy_volume,log_loss_volume,roc_auc_volume,delta_accuracy,delta_log_loss,delta_roc_auc
0,1,0.516588,0.692167,0.520935,0.510202,0.693072,0.488761,-0.006386,0.000905,-0.032174
1,2,0.529923,0.690788,0.540789,0.509387,0.693014,0.508128,-0.020536,0.002226,-0.032661
2,3,0.515708,0.692314,0.517971,0.521442,0.692815,0.498927,0.005734,0.000501,-0.019044
3,4,0.519447,0.691120,0.528017,0.520694,0.692715,0.497695,0.001247,0.001595,-0.030321


In [33]:
comparison_summary_volume_raw = comparison_volume_vs_raw.drop(
    columns="fold"
).mean(numeric_only=True)

comparison_summary_volume_raw

accuracy_raw       0.520416
log_loss_raw       0.691597
roc_auc_raw        0.526928
accuracy_volume    0.515431
log_loss_volume    0.692904
roc_auc_volume     0.498378
delta_accuracy    -0.004985
delta_log_loss     0.001307
delta_roc_auc     -0.028550
dtype: float64

## Interprétation des résultats — Signed Volume

### Hypothèse

L'hypothèse testée est que le `SIGNED_VOLUME` apporte une information complémentaire aux rendements historiques.

Contrairement aux features construites uniquement à partir des `RET_i`, le volume signé renseigne sur le contexte d'activité de marché associé à une allocation. Nous cherchons donc à savoir si le régime récent de volume, ou son écart par rapport au régime long terme, contient un signal exploitable par une régression logistique.

### Résultats observés

Les résultats montrent que les features `volume_3`, `volume_20` et `delta_volume_3_20` ne permettent pas d'améliorer la régression logistique de référence entraînée sur les rendements bruts.

En moyenne, l'accuracy diminue légèrement par rapport au modèle raw returns. La Log-Loss augmente également, ce qui indique une moins bonne qualité probabiliste. Le ROC-AUC baisse fortement, ce qui suggère que le modèle utilisant uniquement ces features de volume discrimine moins bien les classes positives et négatives.

### Analyse économique

Le résultat ne signifie pas nécessairement que le `SIGNED_VOLUME` ne contient aucune information économique.

Il indique plutôt que cette représentation simple du volume, basée uniquement sur des moyennes court terme et long terme, ne permet pas à une régression logistique d'extraire un signal directionnel robuste.

Le volume semble davantage être une variable de contexte qu'une variable directionnelle autonome. Il pourrait donc devenir plus pertinent lorsqu'il est combiné aux rendements, par exemple pour qualifier la crédibilité d'un mouvement de prix.

### Décision

**Décision : rejeter dans cette forme actuelle.**

### Justification

La famille `SIGNED_VOLUME` ajoute une nouvelle source d'information, mais les résultats locaux ne montrent pas de gain robuste contre la régression logistique sur rendements bruts.

Les trois métriques principales évoluent défavorablement : l'accuracy moyenne diminue, la Log-Loss augmente et le ROC-AUC baisse.

### Risque restant

Le phénomène économique associé au volume n'est pas rejeté. Seule cette représentation simple du volume est rejetée.

Une reformulation future pourrait tester des interactions entre rendements et volumes, mais cette piste n'est pas retenue dans cette première phase afin de ne pas mélanger les hypothèses.

In [36]:
feature_engineering_summary = pd.DataFrame(
    [
        {
            "feature_set": "Raw returns",
            "hypothesis": "Historical returns contain directional information.",
            "mean_accuracy": logistic_regression_raw_results["accuracy"].mean(),
            "std_accuracy": logistic_regression_raw_results["accuracy"].std(),
            "mean_roc_auc": logistic_regression_raw_results["roc_auc"].mean(),
            "mean_log_loss": logistic_regression_raw_results["log_loss"].mean(),
            "folds_beating_raw": None,
            "decision": "Reference",
            "justification": "Benchmark model using RET_1 to RET_20.",
        },
        {
            "feature_set": "Momentum",
            "hypothesis": "Recent trends persist across short and long horizons.",
            "mean_accuracy": logistic_regression_momentum_results["accuracy"].mean(),
            "std_accuracy": logistic_regression_momentum_results["accuracy"].std(),
            "mean_roc_auc": logistic_regression_momentum_results["roc_auc"].mean(),
            "mean_log_loss": logistic_regression_momentum_results["log_loss"].mean(),
            "folds_beating_raw": (comparison_summary_momentum_volatility_ratio_raw["delta_accuracy"] > 0).sum(),
            "decision": "Reject",
            "justification": "No robust improvement over raw returns.",
        },
        {
            "feature_set": "Volatility",
            "hypothesis": "Realized volatility captures uncertainty and noise.",
            "mean_accuracy": logistic_regression_volatility_results["accuracy"].mean(),
            "std_accuracy": logistic_regression_volatility_results["accuracy"].std(),
            "mean_roc_auc": logistic_regression_volatility_results["roc_auc"].mean(),
            "mean_log_loss": logistic_regression_volatility_results["log_loss"].mean(),
            "folds_beating_raw": (comparison_volatility_vs_raw["delta_accuracy"] > 0).sum(),
            "decision": "Reject",
            "justification": "Lower ROC-AUC and no stable gain.",
        },
        {
            "feature_set": "Momentum / Volatility ratio",
            "hypothesis": "Risk-adjusted momentum is more informative than raw momentum.",
            "mean_accuracy": logistic_regression_momentum_volatility_ratio_results["accuracy"].mean(),
            "std_accuracy": logistic_regression_momentum_volatility_ratio_results["accuracy"].std(),
            "mean_roc_auc": logistic_regression_momentum_volatility_ratio_results["roc_auc"].mean(),
            "mean_log_loss": logistic_regression_momentum_volatility_ratio_results["log_loss"].mean(),
            "folds_beating_raw": (comparison_momentum_volatility_ratio_vs_raw["delta_accuracy"] > 0).sum(),
            "decision": "Reject",
            "justification": "Risk-adjusted representation does not improve the linear model.",
        },
        {
            "feature_set": "Signed Volume",
            "hypothesis": "Market activity provides complementary information.",
            "mean_accuracy": logistic_regression_volume_results["accuracy"].mean(),
            "std_accuracy": logistic_regression_volume_results["accuracy"].std(),
            "mean_roc_auc": logistic_regression_volume_results["roc_auc"].mean(),
            "mean_log_loss": logistic_regression_volume_results["log_loss"].mean(),
            "folds_beating_raw": (comparison_volume_vs_raw["delta_accuracy"] > 0).sum(),
            "decision": "Reject",
            "justification": "Volume-only representation degrades ROC-AUC and Log-Loss.",
        },
    ]
)

feature_engineering_summary

,feature_set,hypothesis,mean_accuracy,std_accuracy,mean_roc_auc,mean_log_loss,folds_beating_raw,decision,justification
0,Raw returns,Historical returns contain directional informa...,0.520416,0.006536,0.526928,0.691597,NaN,Reference,Benchmark model using RET_1 to RET_20.
1,Momentum,Recent trends persist across short and long ho...,0.515897,0.008216,0.517147,0.692404,0.0,Reject,No robust improvement over raw returns.
2,Volatility,Realized volatility captures uncertainty and n...,0.513834,0.008859,0.497026,0.692892,2.0,Reject,Lower ROC-AUC and no stable gain.
3,Momentum / Volatility ratio,Risk-adjusted momentum is more informative tha...,0.513164,0.006486,0.513089,0.692549,1.0,Reject,Risk-adjusted representation does not improve ...
4,Signed Volume,Market activity provides complementary informa...,0.515431,0.006525,0.498378,0.692904,2.0,Reject,Volume-only representation degrades ROC-AUC an...


# Conclusion du Feature Engineering

Cette étape avait pour objectif d'évaluer si des représentations plus informatives des variables historiques pouvaient améliorer les performances d'une régression logistique, tout en conservant exactement le même protocole expérimental.

Quatre familles de features ont été testées :

- le momentum multi-horizon ;
- la volatilité réalisée ;
- le ratio momentum / volatilité ;
- le signed volume multi-horizon.

Chaque famille a été construite à partir d'une hypothèse économique, évaluée sur les mêmes folds temporels, puis comparée à la régression logistique de référence entraînée sur les rendements bruts `RET_1` à `RET_20`.

Aucune des familles testées n'a montré de gain robuste par rapport au modèle de référence.

Cette conclusion ne signifie pas que les phénomènes économiques étudiés sont inexistants. Elle indique uniquement que les représentations proposées ne fournissent pas, dans le cadre d'une régression logistique linéaire, un signal suffisamment stable pour être retenues.

La meilleure décision méthodologique est donc de conserver les rendements bruts comme socle de features pour le modèle actuel.

La prochaine étape consistera à entraîner une régression logistique finale sur tout le jeu d'entraînement avec ce feature set de référence, puis à générer une première soumission officielle.

Le score public sera interprété uniquement comme un contrôle externe, et non comme un critère de sélection des features.